# Agent ReAct bằng LangGraph: dựng tay, prebuilt, và so với vòng lặp tự viết của module 16

Module 16 tự viết vòng agent và đo nó trên 24 task của một hiệu sách tự tạo. Bài này dựng lại agent đó bằng
LangGraph: tool sinh từ hàm Python bằng `@tool`, node `agent` gọi model đã `bind_tools`, `ToolNode` chạy tool, một cạnh
có điều kiện quyết định dừng hay đi tiếp, ngân sách bước nằm trong state. Rồi chạy cùng 24 task với `qwen2.5:3b` để
so: agent dựng tay, agent prebuilt của LangGraph, và các con số của module 16 (bảng ở mục 2 của module đó).

Câu hỏi của bài không phải "framework có tốt không" mà là: framework **đổi** cái gì (số dòng code, chỗ để cắm
checkpoint, streaming, human-in-the-loop) và **không đổi** cái gì (model vẫn là model, giao thức gọi tool vẫn quyết định
tỉ lệ đúng).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) gọi Ollama, in bảng; test bỏ qua chúng.
- Cần Ollama với `qwen2.5:3b`. Các cell `explore` gọi model khoảng {N_CALLS} lượt, chạy khoảng {T_TOTAL} phút trên PC
  của track (GPU dùng chung).
- Chấm: `uv run pytest tracks/ai-engineer/modules/17-langchain-langgraph/exercises/intermediate/01-langgraph-react -q`

In [ ]:
# explore: đưa gốc repo vào sys.path (test chạy từ gốc repo nên không cần cell này)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

In [ ]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")

import json
import math
import operator
import re
import time
import unicodedata
from typing import Annotated

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import BaseTool, tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode

import bookstore

## 1. Bộ chấm của module 16 (dán lời giải)

Notebook tự chứa: dán `numbers_in`, `check_answer`, `wilson_interval`, `discordant` từ lời giải bài Intermediate 02
module 16 vào cell dưới, giữ nguyên chữ ký. Test của bài này không chấm cell đó; các cell `explore` dùng nó để chấm
24 task **đúng như module 16 chấm**, để hai bảng so được với nhau.

In [ ]:
def numbers_in(text: str) -> list[float]:
    # TODO: dán lời giải bài Intermediate 02 module 16
    raise NotImplementedError


def check_answer(answer: str | None, expected: list) -> bool:
    # TODO: dán lời giải bài Intermediate 02 module 16
    raise NotImplementedError


def wilson_interval(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    # TODO: dán lời giải bài Intermediate 02 module 16
    raise NotImplementedError


def discordant(a: dict[str, bool], b: dict[str, bool]) -> tuple[list[str], list[str]]:
    # TODO: dán lời giải bài Intermediate 02 module 16
    raise NotImplementedError

## 2. Tool từ hàm Python

Bài Basic 01 module 16 tự sinh JSON Schema từ chữ ký hàm. LangChain làm việc đó bằng `tool(fn, parse_docstring=True)`:
tên là tên hàm, mô tả là đoạn đầu docstring, mô tả từng tham số lấy từ mục `Args:` (kiểu Google), `Literal` thành
`enum`.

- `make_tools(functions) -> list[BaseTool]`: một tool cho mỗi hàm, giữ thứ tự, mô tả tham số từ docstring.
- `format_tool_error(exc) -> str`: thông điệp observation khi tool ném lỗi, đúng dạng `"ERROR: <TênLỗi>: <thông
  điệp>"` (cùng tiền tố `ERROR` với module 16). Hàm này sẽ được truyền cho `ToolNode` ở mục 3.

In [ ]:
def make_tools(functions) -> list[BaseTool]:
    # TODO
    raise NotImplementedError


def format_tool_error(exc: Exception) -> str:
    # TODO
    raise NotImplementedError

## 3. Agent dựng tay

```mermaid
flowchart LR
    S((START)) --> agent
    agent -->|"stopped rỗng: còn tool call và còn ngân sách"| tools
    agent -->|"stopped = answer hoặc max_steps"| E((END))
    tools --> agent
```

`AgentState` (đã có) là `MessagesState` thêm `llm_calls` (cộng dồn) và `stopped` (ghi đè).

- `build_agent(llm, tools, *, system=NATIVE_SYSTEM, max_steps=8)`: đồ thị như sơ đồ, `compile()` không checkpointer.
  - Node `agent` gửi `[SystemMessage(system), *messages]` cho `llm.bind_tools(tools)`; system prompt **không** được ghi
    vào state. Trả message của model, `llm_calls: 1`, và `stopped`: `"answer"` nếu không có tool call; `"max_steps"`
    nếu có tool call nhưng đây đã là lượt gọi thứ `max_steps`; ngược lại `""`.
  - Node `tools` là `ToolNode(tools, handle_tool_errors=format_tool_error)`: lỗi tool thành observation, agent đi tiếp.
- `run_task(app, question, *, recursion_limit=50) -> dict`: invoke với message người dùng, `llm_calls=0`,
  `stopped=""` và `recursion_limit` trong config. Trả `{"answer": <nội dung message cuối nếu stopped == "answer",
  ngược lại None>, "stopped", "llm_calls", "messages"}`.

Ngân sách nằm trong state và trong code định tuyến, không phải trong `recursion_limit`: giới hạn đó (mặc định 10 007
superstep ở LangGraph 1.2.16) là lưới an toàn, chạm nó thì `invoke` ném `GraphRecursionError` và bạn mất cả trace.

In [ ]:
NATIVE_SYSTEM = ("Bạn là trợ lý của một hiệu sách. Dùng tool khi cần số liệu; số liệu phải lấy từ kết quả tool. "
                 "Trả lời tiếng Việt, ngắn gọn.")


class AgentState(MessagesState):
    llm_calls: Annotated[int, operator.add]
    stopped: str

In [ ]:
def build_agent(llm, tools: list[BaseTool], *, system: str = NATIVE_SYSTEM, max_steps: int = 8):
    # TODO
    raise NotImplementedError


def run_task(app, question: str, *, recursion_limit: int = 50) -> dict:
    # TODO
    raise NotImplementedError

## 4. Trace kiểu module 16

Lịch sử message của LangGraph **là** trace, chỉ khác hình dạng. `to_steps(messages) -> list[dict]` đổi nó về dạng bước
của module 16 để dùng lại `classify_failure` và các bảng của module đó:

- Mỗi `AIMessage` là một bước; `index` đếm từ 0 theo thứ tự các AIMessage.
- AIMessage không có tool call: một dict `{"index", "kind": "final", "tool": None, "args": None, "observation": None,
  "ok": None}`.
- AIMessage có tool call: **mỗi** tool call một dict `kind: "action"` cùng `index`, `tool`, `args`, `observation` là
  nội dung (`str`) của `ToolMessage` có `tool_call_id` trùng, `ok` là `status != "error"` của nó; không có ToolMessage
  trả lời (hết ngân sách giữa chừng) thì `observation` và `ok` là `None`.
- Bỏ qua message người dùng và system.

In [ ]:
def to_steps(messages) -> list[dict]:
    # TODO
    raise NotImplementedError

## 5. Chạy thật trên 24 task

Cùng model, cùng `temperature` 0, `seed` 0, cùng system prompt `NATIVE_SYSTEM` và tối đa 8 lượt gọi model như cách
native của module 16. `ChatOllama` của `langchain-ollama` gọi `/api/chat` của Ollama với `tools`, tức đúng giao thức
native. Một callback đo thời gian từng lượt gọi model để tách thời gian của **framework** khỏi thời gian của **model**.

In [ ]:
# explore: model, tool, callback đo thời gian, hàm chạy một cấu hình trên 24 task
import warnings

import pandas as pd
from langchain_core.callbacks import BaseCallbackHandler
from langchain_ollama import ChatOllama
from langgraph.errors import GraphRecursionError

llm = ChatOllama(model="qwen2.5:3b", temperature=0, seed=0)
tools = make_tools(bookstore.TOOLS)
print([t.name for t in tools])
print(json.dumps(tools[0].tool_call_schema.model_json_schema(), ensure_ascii=False)[:300])


class LLMTimer(BaseCallbackHandler):
    def __init__(self):
        self.started, self.seconds, self.calls = {}, 0.0, 0

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        self.started[run_id] = time.perf_counter()
        self.calls += 1

    def on_llm_end(self, response, *, run_id, **kwargs):
        self.seconds += time.perf_counter() - self.started.pop(run_id)


def evaluate(name, invoke):
    rows = []
    for task_id, group, question, expected in bookstore.TASKS:
        timer = LLMTimer()
        start = time.perf_counter()
        try:
            out = invoke(question, timer)
        except GraphRecursionError:
            out = {"answer": None, "stopped": "recursion_limit", "messages": []}
        except Exception as exc:
            out = {"answer": None, "stopped": f"crash: {type(exc).__name__}", "messages": []}
        seconds = time.perf_counter() - start
        steps = to_steps(out["messages"])
        rows.append(dict(mode=name, id=task_id, group=group, ok=check_answer(out["answer"], expected),
                         stopped=out["stopped"], llm_calls=timer.calls, seconds=seconds,
                         framework_ms=1000 * (seconds - timer.seconds) / max(timer.calls, 1),
                         actions=sum(s["kind"] == "action" for s in steps), answer=out["answer"] or "", steps=steps))
        r = rows[-1]
        print(f"{name:16s} {task_id} {'ĐÚNG' if r['ok'] else 'sai '} {r['stopped']:24s} {r['llm_calls']} lượt "
              f"{seconds:5.1f}s | {r['answer'][:70]!r}")
    return rows

In [ ]:
# explore: agent dựng tay trên 24 task (khoảng {T_HAND} phút)
hand = build_agent(llm, tools, max_steps=8)


def invoke_hand(question, timer):
    out = hand.invoke({"messages": [HumanMessage(question)], "llm_calls": 0, "stopped": ""},
                      {"recursion_limit": 50, "callbacks": [timer]})
    return {"answer": out["messages"][-1].content if out["stopped"] == "answer" else None,
            "stopped": out["stopped"], "messages": out["messages"]}


rows = evaluate("dựng tay", invoke_hand)

In [ ]:
# explore: agent prebuilt create_react_agent, mặc định và có handle_tool_errors (khoảng {T_PRE} phút)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    from langgraph.prebuilt import create_react_agent

    prebuilt_default = create_react_agent(llm, tools, prompt=NATIVE_SYSTEM)
    prebuilt_fixed = create_react_agent(llm, ToolNode(tools, handle_tool_errors=format_tool_error), prompt=NATIVE_SYSTEM)
print({type(w.message).__name__: str(w.message)[:160] for w in caught})


def make_invoke(agent):
    def invoke(question, timer):
        out = agent.invoke({"messages": [HumanMessage(question)]}, {"recursion_limit": 16, "callbacks": [timer]})
        last = out["messages"][-1]
        done = not getattr(last, "tool_calls", None) and "need more steps" not in last.content
        return {"answer": last.content if done else None, "stopped": "answer" if done else "max_steps",
                "messages": out["messages"]}
    return invoke


rows += evaluate("prebuilt mặc định", make_invoke(prebuilt_default))
rows += evaluate("prebuilt + lỗi", make_invoke(prebuilt_fixed))

In [ ]:
# explore: bảng tổng hợp, so theo cặp, thời gian framework
df = pd.DataFrame(rows)
summary = []
for mode, g in df.groupby("mode", sort=False):
    k, n = int(g.ok.sum()), len(g)
    lo, hi = wilson_interval(k, n)
    summary.append(dict(cach=mode, dung=f"{k}/{n}", ci95=f"[{lo:.2f}, {hi:.2f}]", luot_llm_tb=round(g.llm_calls.mean(), 2),
                        giay_trung_vi=round(g.seconds.median(), 1), framework_ms_moi_luot=round(g.framework_ms.median(), 1),
                        sai_sau_toi_da_1_tool=int(((~g.ok) & (g.actions <= 1)).sum()),
                        dung_lai=g.stopped.value_counts().to_dict()))
display(pd.DataFrame(summary))
by_mode = {m: dict(zip(g.id, g.ok)) for m, g in df.groupby("mode", sort=False)}
print("dựng tay vs prebuilt + lỗi:", discordant(by_mode["dựng tay"], by_mode["prebuilt + lỗi"]))
print(df.pivot_table(index="group", columns="mode", values="ok", aggfunc="sum", sort=False))

In [ ]:
# explore: streaming. "updates" cho từng bước của đồ thị, "messages" cho từng token của model
question = bookstore.TASKS[12][2]
for chunk in hand.stream({"messages": [HumanMessage(question)], "llm_calls": 0, "stopped": ""},
                         {"recursion_limit": 50}, stream_mode="updates"):
    for node, update in chunk.items():
        last = update["messages"][-1]
        print(f"[{node}]", getattr(last, "tool_calls", None) or str(last.content)[:120])

start, first = time.perf_counter(), None
for token, meta in hand.stream({"messages": [HumanMessage(bookstore.TASKS[1][2])], "llm_calls": 0, "stopped": ""},
                               {"recursion_limit": 50}, stream_mode="messages"):
    if meta["langgraph_node"] == "agent" and token.content:
        first = first or time.perf_counter() - start
        print(token.content, end="", flush=True)
print(f"\ntoken đầu tiên sau {first:.2f} s, cả câu {time.perf_counter() - start:.2f} s")

## Đọc kết quả

Tự trả lời trước khi làm trắc nghiệm:

- Agent dựng tay và prebuilt (có xử lý lỗi) dùng cùng giao thức native: tỉ lệ đúng của chúng gần nhau tới đâu, và gần
  cách nào của module 16 hơn: ReAct văn bản hay native? Vì sao đổi framework không đổi được điều đó?
- Prebuilt **mặc định** hỏng ở những task nào, kiểu hỏng gì? Đọc lại mục "ToolNode" ở lý thuyết.
- `framework_ms_moi_luot` so với thời gian một lượt model: framework tốn bao nhiêu phần trăm?
- Các task sai có cùng kiểu với module 16 không (dừng sau nhiều nhất một tool, hỏi lại người dùng thay vì gọi tool)?
- Với streaming, người dùng thấy chữ đầu tiên sau bao lâu so với cả câu?

Sau đó làm phần trắc nghiệm của bài trong app.